<a href="https://colab.research.google.com/github/jaysulk/Poisson-GENERIC-Neural-Operators/blob/main/dg_check_2Dheat_fno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DG solve on 2D heat with the FNO backbone (seed 0)

Same definitions as the previous check notebooks (split predictor in `step_dg`). One training run of 2D heat / FNO / PG-phys, evaluated under the split step and the DG step with 3/6/12 iterations, to decide whether the DG non-convergence seen on the Transolver models is backbone-specific. ~5 min on a T4; result written to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import math, time, itertools, copy
from dataclasses import dataclass, asdict, field
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cuda.matmul.allow_tf32 = False   # keep the structural residuals honest
torch.backends.cudnn.allow_tf32 = False
print('device:', DEV, '| torch', torch.__version__)

def seed_all(s):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def sdims(dim):            # spatial dims of a (B, C, *grid) tensor
    return tuple(range(-dim, 0))

def ip(a, b):              # grid-measure inner product: sum over channels and space / number of grid points -> (B,)
    # NOTE: dividing by the number of grid points only (not channels) keeps the measure independent of the channel
    # count, so Q(u) on the p observed channels and E(z) on the p+1 channels of z use the SAME inner product.  With a
    # per-channel mean, the Riesz-scaled gradient of Q came out (p+1)/p x too large (3x for the wave), the exact
    # exponential integrated only 1/(p+1) of the linear flow and the learned symbols settled at 1/2 (scalar) and 1/3 (wave).
    return (a * b).flatten(1).sum(1) / a[0, 0].numel()

def ip_pointwise(a, b):    # inner product over the channel axis only -> (B, *grid)
    return (a * b).sum(1)

def nrm(a):
    return ip(a, a).sqrt()

def neg_index(a, dim):     # a[..., k] -> a[..., (-k) mod n] on the last `dim` axes (fftn layout)
    d = sdims(dim)
    return torch.roll(torch.flip(a, d), shifts=(1,) * dim, dims=d)

device: cuda | torch 2.11.0+cu128


In [ ]:
@dataclass
class Cfg:
    # problem
    dim: int = 1
    pde: str = 'heat'            # heat | advection | burgers | wave
    nx: int = 64
    n_traj: int = 120
    T: int = 20                  # saved frames per trajectory (beyond t=0)
    dt: float = 0.1              # frame spacing
    kmax: int = 8                # band limit of the initial conditions
    # model
    model: str = 'pg'            # 'pg' (Poisson-GENERIC) | 'base' (unconstrained residual operator)
    backbone: str = 'fno'        # fno | transolver | cno
    width: int = 32              # backbone width (baseline) / functional width (pg)
    modes: int = 12              # FNO modes
    layers: int = 3
    norm: bool = False           # InstanceNorm inside FNO/CNO blocks (paper: needed in 2D, harmful in 1D)
    m_L: int = 16                # band of the Poisson multiplier  (|k|_inf <= m_L, k != 0)
    m_M: int = 16                # band of the dissipative multiplier
    K_mode: str = 'complement'   # kernel band the entropy sees: 'complement' (k=0 and |k|>m_L) | 'mean' (k=0 only)
    s_init: str = 'encoder'      # latent entropy density init: 'encoder' | 'zero'
    op_init: float = 0.1         # init scale of the multiplier symbols (too small -> model starts in the identity basin)
    energy: str = 'learned'      # 'learned': E = h(mean g(u,s)) | 'physical': E = Q(u) + mean e(s), Q fixed, e >= 0 learned
    entropy: str = 'learned'     # 'learned': Casimir functional of (Pi_K u, s) | 'integral': S = mean(s) (exact Casimir)
    m_type: str = 'constant'     # 'constant': D_M multiplier | 'state': A(z)A(z)^T with A = D_M o Gamma(z), Gamma a pointwise matrix field
    n_sub: int = 1               # split integrator: sub-steps for the explicit dissipative remainder (stability: coef*k_max^2/n_sub < 2)
    flux_init: bool = True       # state-M: initialize D_M as gradient/divergence flux slots (model starts as a nonlinear diffusion)
    flux_scale: float = 0.05     # scale of that init; explicit Euler on the dissipative part is stable only if coef*k_max^2 < 2
    lie_poisson: bool = False    # scalar fields: add lambda*(u D + D u), D = sum_a d_a, to L (Lie-Poisson, Poisson; compatible with D):
                                 #   nonlinear transport (Burgers = -(1/3)(uD+Du) delta/du 1/2||u||^2) with a QUADRATIC energy
    phi_gauge: str = 'proj'    # 'taylor': subtract phi's degree-2 Taylor polynomial at 0 (removes the quadratic gauge) | 'none': phi >= 0
    conserve_mass: str = 'auto'  # 'auto': for the scalar conservation laws (heat/advection/burgers) M gets no local (zeroth-order) slot on u,
                                 #   so int u is conserved and there is no Rayleigh-type damping of a conserved density | 'yes' | 'no'
    # training
    epochs: int = 40
    batch: int = 16
    horizon: int = 4             # training rollout horizon
    lr: float = 1e-3
    lr_op: float = 3e-2          # learning rate of the operator symbols a(k), B(k): the true advection symbol is a(k) = -c k dt
                                 #   (= -0.8 at k=8), unreachable from a 0.1 init in ~100 Adam steps at lr 1e-3
    wd: float = 1e-4
    # evaluation
    n_roll: int = 10
    integrator: str = 'euler'    # evaluation integrator: 'euler' | 'dg' (discrete gradient, exact E/S) | 'split' (exact unitary step
                                 #   for the linear reversible part exp(L Hess Q) + Euler for the rest; physical variant only)
    train_integrator: str = 'euler'   # training integrator ('dg' is exact but ~dg_iters x slower)
    dg_iters: int = 6
    seed: int = 0

PDE_PARAMS = {
    'heat':      dict(nu=0.05),
    'advection': dict(c=1.0),
    'burgers':   dict(nu=0.02),
    'wave':      dict(c=1.0, gamma=0.3),
}
def n_fields(pde):           # observed channels p
    return 2 if pde == 'wave' else 1

In [ ]:
class Spectral:
    """Spectral derivatives on the periodic box [0, 2pi)^d with integer wavenumbers."""
    def __init__(self, nx, dim, device=DEV, dtype=torch.float64):
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=device, dtype=dtype)
        self.ks = torch.meshgrid(*([k] * dim), indexing='ij')
        self.k2 = sum(kk ** 2 for kk in self.ks)
        kinf = torch.stack([kk.abs() for kk in self.ks]).amax(0)
        self.dealias_mask = (kinf < nx / 3.0)          # 2/3 rule
        self.dim = dim; self.d = sdims(dim)
    def fft(self, u):  return torch.fft.fftn(u, dim=self.d)
    def ifft(self, uh): return torch.fft.ifftn(uh, dim=self.d).real
    def deriv(self, u, axis):
        return self.ifft(1j * self.ks[axis] * self.fft(u))
    def lap(self, u):
        return self.ifft(-self.k2 * self.fft(u))
    def dealias(self, u):
        return self.ifft(self.dealias_mask * self.fft(u))

def pde_rhs(pde, u, sp, prm):
    """u: (B, p, *grid) float64. Returns du/dt."""
    if pde == 'heat':
        return prm['nu'] * sp.lap(u)
    if pde == 'advection':
        return -prm['c'] * sum(sp.deriv(u, a) for a in range(sp.dim))
    if pde == 'burgers':
        adv = sum(sp.deriv(u, a) for a in range(sp.dim))
        return -sp.dealias(u * adv) + prm['nu'] * sp.lap(u)
    if pde == 'wave':
        q, v = u[:, :1], u[:, 1:]
        return torch.cat([v, prm['c'] ** 2 * sp.lap(q) - prm['gamma'] * v], 1)
    raise ValueError(pde)

def rk4(f, u, dt):
    k1 = f(u); k2 = f(u + 0.5 * dt * k1); k3 = f(u + 0.5 * dt * k2); k4 = f(u + dt * k3)
    return u + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def random_field(n, nx, dim, kmax, device=DEV, dtype=torch.float64):
    """Band-limited random field with a mild red spectrum, unit RMS."""
    sp = Spectral(nx, dim, device, dtype)
    w = torch.randn(n, 1, *([nx] * dim), device=device, dtype=dtype)
    kinf = torch.stack([kk.abs() for kk in sp.ks]).amax(0)
    mask = (kinf <= kmax) & (kinf > 0)
    amp = 1.0 / (1.0 + sp.k2) ** 0.75
    u = sp.ifft(sp.fft(w) * mask * amp)
    return u / u.flatten(1).std(1).view(n, *([1] * (dim + 1)))

def n_substeps(cfg, nx):
    """RK4 sub-steps per frame from the linear stability limit at this resolution (nu k^2 dt, c k dt < ~2.8)."""
    prm = PDE_PARAMS[cfg.pde]; kmax = nx / 2
    if cfg.pde in ('heat', 'burgers'):
        dt_max = min(0.02, 2.0 / (prm['nu'] * cfg.dim * kmax ** 2))
    else:
        dt_max = min(0.02, 2.0 / (prm['c'] * cfg.dim * kmax))
    return max(1, int(math.ceil(cfg.dt / dt_max)))

def gen_data(cfg, device=DEV):
    """Returns (n_traj, T+1, p, *grid) float32 trajectories."""
    seed_all(cfg.seed)
    p = n_fields(cfg.pde); prm = PDE_PARAMS[cfg.pde]
    sp = Spectral(cfg.nx, cfg.dim, device)
    u = random_field(cfg.n_traj, cfg.nx, cfg.dim, cfg.kmax, device)
    if cfg.pde == 'wave':
        v = 0.5 * random_field(cfg.n_traj, cfg.nx, cfg.dim, cfg.kmax, device)
        u = torch.cat([u, v], 1)
    n_sub = n_substeps(cfg, cfg.nx); dt_sub = cfg.dt / n_sub
    f = lambda x: pde_rhs(cfg.pde, x, sp, prm)
    frames = [u]
    with torch.no_grad():
        for _ in range(cfg.T):
            for _ in range(n_sub):
                u = rk4(f, u, dt_sub)
            frames.append(u)
    return torch.stack(frames, 1).float()

def split_data(data, frac=0.8):
    n = int(frac * data.shape[0])
    return data[:n], data[n:]

In [ ]:
def ConvNd(dim):  return {1: nn.Conv1d, 2: nn.Conv2d}[dim]
def INormNd(dim): return {1: nn.InstanceNorm1d, 2: nn.InstanceNorm2d}[dim]

def coord_grid(x):
    """(B, C, *grid) -> (B, dim, *grid) coordinates in [0,1)."""
    B, dim = x.shape[0], x.dim() - 2
    axes = [torch.linspace(0, 1, x.shape[2 + i] + 1, device=x.device, dtype=x.dtype)[:-1] for i in range(dim)]
    g = torch.stack(torch.meshgrid(*axes, indexing='ij'), 0)
    return g.unsqueeze(0).expand(B, *g.shape)

class SpectralConv(nn.Module):
    def __init__(self, dim, cin, cout, modes):
        super().__init__()
        self.dim, self.m = dim, modes
        scale = 1.0 / (cin * cout)
        shape = (cin, cout) + (modes,) * dim + (2,)
        self.w1 = nn.Parameter(scale * torch.randn(*shape))
        if dim == 2: self.w2 = nn.Parameter(scale * torch.randn(*shape))
    def forward(self, x):
        m = self.m
        if self.dim == 1:
            xf = torch.fft.rfft(x)
            out = torch.zeros(x.shape[0], self.w1.shape[1], xf.shape[-1], dtype=xf.dtype, device=x.device)
            out[:, :, :m] = torch.einsum('bim,iom->bom', xf[:, :, :m], torch.view_as_complex(self.w1))
            return torch.fft.irfft(out, n=x.shape[-1])
        xf = torch.fft.rfft2(x)
        out = torch.zeros(x.shape[0], self.w1.shape[1], *xf.shape[-2:], dtype=xf.dtype, device=x.device)
        out[:, :, :m, :m] = torch.einsum('bixy,ioxy->boxy', xf[:, :, :m, :m], torch.view_as_complex(self.w1))
        out[:, :, -m:, :m] = torch.einsum('bixy,ioxy->boxy', xf[:, :, -m:, :m], torch.view_as_complex(self.w2))
        return torch.fft.irfft2(out, s=x.shape[-2:])

class FNO(nn.Module):
    """Fourier Neural Operator (Li et al., 2021), dimension-generic, optional InstanceNorm per block."""
    def __init__(self, dim, cin, cout, width=32, modes=12, layers=3, norm=False):
        super().__init__()
        C = ConvNd(dim)
        self.lift = C(cin + dim, width, 1)
        self.specs = nn.ModuleList([SpectralConv(dim, width, width, modes) for _ in range(layers)])
        self.ws = nn.ModuleList([C(width, width, 1) for _ in range(layers)])
        self.norms = nn.ModuleList([INormNd(dim)(width, affine=True) if norm else nn.Identity() for _ in range(layers)])
        self.proj = nn.Sequential(C(width, 2 * width, 1), nn.GELU(), C(2 * width, cout, 1))
    def forward(self, x):
        x = self.lift(torch.cat([x, coord_grid(x)], 1))
        for s, w, n in zip(self.specs, self.ws, self.norms):
            x = F.gelu(n(s(x) + w(x)))
        return self.proj(x)

class PhysicsAttention(nn.Module):
    """Transolver physics-attention (Wu et al., ICML 2024): tokens -> M learned slices -> attention -> back."""
    def __init__(self, C, heads=4, slices=32):
        super().__init__()
        self.h, self.s, self.ch = heads, slices, C // heads
        self.in_proj = nn.Linear(C, C)
        self.slice_proj = nn.Linear(C, heads * slices)
        self.temp = nn.Parameter(torch.tensor(0.5))
        self.qkv = nn.Linear(self.ch, 3 * self.ch)
        self.out = nn.Linear(C, C)
    def forward(self, x):                       # x: (B, N, C)
        B, N, C = x.shape
        fx = self.in_proj(x).view(B, N, self.h, self.ch)
        w = torch.softmax(self.slice_proj(x).view(B, N, self.h, self.s) / self.temp.clamp(min=0.05), dim=-1)
        tok = torch.einsum('bnhs,bnhc->bhsc', w, fx) / (w.sum(1).unsqueeze(-1) + 1e-6)   # (B,h,S,ch)
        q, k, v = self.qkv(tok).chunk(3, -1)
        att = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(self.ch), -1)
        tok = att @ v
        y = torch.einsum('bnhs,bhsc->bnhc', w, tok).reshape(B, N, C)
        return self.out(y)

class Transolver(nn.Module):
    """Transolver physics attention (Wu et al., ICML 2024) with structured-mesh-style local mixing (v11): a kernel-3
    circular depthwise convolution on the grid precedes each block, so every block sees a local stencil as well as the
    global slice tokens. The v10 version had no local operator at all (every projection a Linear on flattened tokens):
    as a residual baseline it could not represent a one-cell shift, sat at the persistence level of the training loss on
    advection/wave/Burgers and learned a point-wise damping instead. Cost: layers * width * (kernel**dim + 1) parameters."""
    def __init__(self, dim, cin, cout, width=64, layers=3, heads=4, slices=32, kernel=3):
        super().__init__()
        self.dim = dim
        self.lift = nn.Sequential(nn.Linear(cin + dim, width), nn.GELU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList()
        for _ in range(layers):
            self.blocks.append(nn.ModuleDict(dict(
                ln1=nn.LayerNorm(width), att=PhysicsAttention(width, heads, slices),
                ln2=nn.LayerNorm(width),
                mlp=nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Linear(2 * width, width)))))
        C = ConvNd(dim)
        self.mix = nn.ModuleList([C(width, width, kernel, padding=kernel // 2, padding_mode='circular', groups=width)
                                  for _ in range(layers)])
        self.proj = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, cout))
    def forward(self, x):
        B, C, *grid = x.shape
        t = torch.cat([x, coord_grid(x)], 1).flatten(2).transpose(1, 2)      # (B, N, C+dim)
        t = self.lift(t)
        for b, m in zip(self.blocks, self.mix):
            t = t + m(t.transpose(1, 2).reshape(B, -1, *grid)).flatten(2).transpose(1, 2)   # local mixing on the grid
            t = t + b['att'](b['ln1'](t))
            t = t + b['mlp'](b['ln2'](t))
        y = self.proj(t).transpose(1, 2).reshape(B, -1, *grid)
        return y

class CNO(nn.Module):
    """CNO-style periodic convolutional operator (Raonic et al., 2023), simplified: 2-level U-shape,
    circular padding, GELU; no alias-free up/down-sampled activations (hence '-lite')."""
    def __init__(self, dim, cin, cout, width=32, layers=2, norm=False):
        super().__init__()
        C = ConvNd(dim); self.dim = dim
        def block(ci, co):
            mods = [C(ci, co, 3, padding=1, padding_mode='circular')]
            if norm: mods.append(INormNd(dim)(co, affine=True))
            mods.append(nn.GELU())
            return nn.Sequential(*mods)
        self.lift = C(cin + dim, width, 1)
        self.enc1 = nn.Sequential(*[block(width, width) for _ in range(layers)])
        self.enc2 = nn.Sequential(block(width, 2 * width), *[block(2 * width, 2 * width) for _ in range(layers - 1)])
        self.dec = nn.Sequential(block(3 * width, width), *[block(width, width) for _ in range(layers - 1)])
        self.proj = C(width, cout, 1)
        self.pool = {1: F.avg_pool1d, 2: F.avg_pool2d}[dim]
        self.mode = {1: 'linear', 2: 'bilinear'}[dim]
    def forward(self, x):
        x = self.lift(torch.cat([x, coord_grid(x)], 1))
        h1 = self.enc1(x)
        h2 = self.enc2(self.pool(h1, 2))
        up = F.interpolate(h2, size=h1.shape[2:], mode=self.mode, align_corners=False)
        return self.proj(self.dec(torch.cat([h1, up], 1)))

def make_backbone(cfg, cin, cout, width=None):
    w = width or cfg.width
    if cfg.backbone == 'fno':
        return FNO(cfg.dim, cin, cout, w, cfg.modes, cfg.layers, cfg.norm)
    if cfg.backbone == 'transolver':
        return Transolver(cfg.dim, cin, cout, max(w, 32), cfg.layers)
    if cfg.backbone == 'cno':
        return CNO(cfg.dim, cin, cout, w, max(cfg.layers - 1, 1), cfg.norm)
    raise ValueError(cfg.backbone)

def zero_last_layer(net):
    last = [m for m in net.modules() if isinstance(m, (nn.Linear, nn.Conv1d, nn.Conv2d))][-1]
    nn.init.zeros_(last.weight); nn.init.zeros_(last.bias)

def count_params(m): return sum(p.numel() for p in m.parameters())

In [ ]:
def _scatter_band(param, nx, dim):
    """param: (2m+1,)*dim tensor indexed by k in [-m..m]  ->  (nx,)*dim tensor in fftn layout."""
    m = (param.shape[-1] - 1) // 2
    assert m < nx // 2, f'band m={m} must be < nx/2={nx//2}'
    idx = torch.arange(-m, m + 1, device=param.device) % nx
    full = torch.zeros(*([nx] * dim), device=param.device, dtype=param.dtype)
    if dim == 1:
        full[idx] = param
    else:
        full[idx[:, None], idx[None, :]] = param
    return full

class PoissonOperator(nn.Module):
    """Constant-coefficient skew Fourier multiplier on the mechanical block -> Poisson (Jacobi trivially).
    Scalar field:  symbol i a(k), a odd, a=0 on the kernel band K.
    Wave (q,v):    canonical J composed with an even, non-negative multiplier d(k) (zero on K)."""
    def __init__(self, dim, p, m, K_mode='complement', canonical=False, init_scale=0.05):
        super().__init__()
        self.dim, self.p, self.m, self.K_mode, self.canonical = dim, p, m, K_mode, canonical
        self.a = nn.Parameter(init_scale * torch.randn(*([2 * m + 1] * dim)))
        self.lam = nn.Parameter(torch.zeros(1))            # Lie-Poisson coefficient (scalar fields only; 0 = off)
        self.use_lp = False
        # KdV pencil: when the Lie-Poisson term is on, the constant part is restricted to alpha1 D + alpha3 D^3 (D = sum_a d_a),
        # which is COMPATIBLE with u D + D u (the bi-Hamiltonian pair of KdV), so the full L is Poisson for all parameters.
        # A generic multiplier a(k) is not compatible with the Lie-Poisson term (Jacobi fails at O(0.1)), see Sec. 7.
        self.alpha = nn.Parameter(torch.tensor([init_scale, 0.0]))
        self._cache = {}
    def symbol(self, nx):
        if self.use_lp:                                    # pencil: i(alpha1 kD - alpha3 kD^3) on the band, kD = sum_a k_a
            k = torch.fft.fftfreq(nx, d=1.0 / nx, device=self.a.device)
            ks = torch.meshgrid(*([k] * self.dim), indexing='ij'); kD = sum(ks)
            kinf = torch.stack([kk.abs() for kk in ks]).amax(0); band = (kinf <= self.m) & (kinf > 0)
            # alpha3 D^3 is compatible too but its k^3 growth wrecks the discrete-gradient fixed point and the explicit
            # remainder at high k; none of the test PDEs is dispersive, so the pencil is restricted to alpha1 D + lambda(uD+Du).
            return 1j * (self.alpha[0] * kD) * band
        full = _scatter_band(self.a, nx, self.dim)
        if self.canonical:
            d2 = full ** 2
            sym = 0.5 * (d2 + neg_index(d2, self.dim))                 # even, >= 0
            sym = sym.clone(); sym[(0,) * self.dim] = 0.0              # kernel contains k=0
            return sym.to(torch.complex64 if full.dtype == torch.float32 else torch.complex128)
        odd = 0.5 * (full - neg_index(full, self.dim))                 # odd  -> real skew operator
        return 1j * odd
    def K_mask(self, nx, device):
        key = (nx, str(device))
        if key not in self._cache:
            k = torch.fft.fftfreq(nx, d=1.0 / nx, device=device)
            ks = torch.meshgrid(*([k] * self.dim), indexing='ij')
            kinf = torch.stack([kk.abs() for kk in ks]).amax(0)
            zero = (kinf == 0)
            mask = zero if self.K_mode == 'mean' else (zero | (kinf > self.m))
            self._cache[key] = mask
        return self._cache[key]
    def project_K(self, u):                      # Pi_K: orthogonal projection onto the kernel band
        d = sdims(self.dim)
        return torch.fft.ifftn(torch.fft.fftn(u, dim=d) * self.K_mask(u.shape[-1], u.device), dim=d).real
    def exp_linear(self, u, cfg, dt=1.0):
        """Exact flow of  u_t = L dQ/du  for the FIXED quadratic energy Q over time dt: a unitary per-mode map,
        exp(dt * L Hess Q).  Scalar: multiply u_hat(k) by exp(i dt a(k)) (|.|=1 -> Q conserved exactly).
        Wave (q,v), Q = 1/2(v^2 + c^2 |grad q|^2):  per-mode 2x2 exponential of [[0, d],[-d c^2 |k|^2, 0]]."""
        d = sdims(self.dim); uf = torch.fft.fftn(u, dim=d); sym = self.symbol(u.shape[-1])
        if not self.canonical:
            return torch.fft.ifftn(uf * torch.exp(dt * sym), dim=d).real
        nx = u.shape[-1]; k = torch.fft.fftfreq(nx, d=1.0 / nx, device=u.device, dtype=u.dtype)
        k2 = sum(kk ** 2 for kk in torch.meshgrid(*([k] * self.dim), indexing='ij'))
        dsym = sym.real; c2 = PDE_PARAMS['wave']['c'] ** 2
        w = dsym * (c2 * k2).sqrt()                                       # oscillation frequency per mode
        cw, sw = torch.cos(dt * w), torch.sin(dt * w)
        ratio = torch.where(w > 0, dsym / w.clamp_min(1e-30), torch.zeros_like(w))   # d/omega (q <- v coupling)
        inv = torch.where(w > 0, w / dsym.clamp_min(1e-30), torch.zeros_like(w))    # omega/d (v <- q coupling)
        q, v = uf[:, 0], uf[:, 1]
        q1 = cw * q + ratio * sw * v
        v1 = -inv * sw * q + cw * v
        return torch.fft.ifftn(torch.stack([q1, v1], 1), dim=d).real
    def lie_poisson(self, u, g):
        """lambda (u D g + D(u g)),  D = sum_a d_a: skew-adjoint on the grid (pointwise products are symmetric, D is skew)
        and a Hamiltonian operator (Lie-Poisson bracket of vector fields along D); the diagnostics of Sec. 7 verify Jacobi
        for the full L = constant multiplier + Lie-Poisson term numerically."""
        d = sdims(self.dim); nx = u.shape[-1]
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=u.device, dtype=u.dtype)
        ks = torch.meshgrid(*([k] * self.dim), indexing='ij'); Dsym = 1j * sum(ks)
        kinf = torch.stack([kk.abs() for kk in ks]).amax(0); mask = (kinf < nx / 3.0)          # 2/3-rule projector P
        D = lambda x: torch.fft.ifftn(torch.fft.fftn(x, dim=d) * Dsym, dim=d).real
        P = lambda x: torch.fft.ifftn(torch.fft.fftn(x, dim=d) * mask, dim=d).real
        # P (u D + D u) P : the sandwich keeps skewness (P symmetric) and removes quadratic aliasing.  The discrete
        # operator is Poisson only in the RESOLVED regime: Jacobi involves triple products, and holds to machine
        # precision when the fields' band b satisfies 3b <= nx/2 (see lie_poisson_jacobi_vs_resolution), but fails at
        # O(0.1) on generic full-spectrum states.  This is why the Lie-Poisson term is OFF by default: with it, exact
        # Jacobi for any state is lost, and Burgers' nonlinear transport is the price (Sec. 6 of the paper).
        u = P(u); g = P(g)
        return self.lam * P(u * D(g) + D(u * g))
    def forward(self, g, u=None):                # g: (B, p, *grid) gradient wrt u  ->  L_u g   (u needed for the LP term)
        d = sdims(self.dim); sym = self.symbol(g.shape[-1])
        mul = lambda x: torch.fft.ifftn(torch.fft.fftn(x, dim=d) * sym, dim=d).real
        if not self.canonical:
            out = mul(g)
            if self.use_lp and u is not None: out = out + self.lie_poisson(u, g)
            return out
        gq, gv = g[:, :1], g[:, 1:]
        return torch.cat([mul(gv), -mul(gq)], 1)  # J D

class DissipativeOperator(nn.Module):
    """M = (I - P_E) D_M (I - P_E), D_M a per-mode Hermitian PSD matrix symbol B(k)B(k)^H on all channels of z."""
    def __init__(self, dim, c, m, init_scale=0.05):
        super().__init__()
        self.dim, self.c, self.m = dim, c, m
        self.B = nn.Parameter(init_scale * torch.randn(c, c, *([2 * m + 1] * dim), 2))
    def symbol(self, nx):
        Bc = torch.view_as_complex(self.B)
        full = torch.stack([torch.stack([_scatter_band(Bc[i, j], nx, self.dim) for j in range(self.c)]) for i in range(self.c)])
        S = torch.einsum('ij...,kj...->ik...', full, full.conj())            # B B^H  (Hermitian PSD per mode)
        return 0.5 * (S + neg_index(S, self.dim).conj())                      # reality: S(-k) = conj S(k)
    def raw(self, v):
        d = sdims(self.dim); S = self.symbol(v.shape[-1])
        vf = torch.fft.fftn(v, dim=d)
        return torch.fft.ifftn(torch.einsum('ij...,bj...->bi...', S, vf), dim=d).real
    @staticmethod
    def proj_off(w, v, tiny=1e-30):              # (I - P_w) v   (exact ratio; guarded only against w == 0)
        coef = ip(w, v) / ip(w, w).clamp_min(tiny)
        return v - coef.view(-1, *([1] * (v.dim() - 1))) * w
    def forward(self, v, w):                     # M v, with degeneracy direction w = dE/dz
        return self.proj_off(w, self.raw(self.proj_off(w, v)))

In [ ]:
class Functional(nn.Module):
    """Scalar functional  F[z] = h( mean_x g(z) ),  g a neural operator backbone. Optional input projector."""
    def __init__(self, cfg, cin, proj=None):
        super().__init__()
        self.net = make_backbone(cfg, cin, cfg.width)
        self.head = nn.Sequential(nn.Linear(cfg.width, 64), nn.GELU(), nn.Linear(64, 1))
        self.proj = proj
    def forward(self, z, p):
        u, s = z[:, :p], z[:, p:]
        if self.proj is not None:
            u = self.proj(u)
        f = self.net(torch.cat([u, s], 1))
        return self.head(f.mean(sdims(z.dim() - 2))).squeeze(-1)

class PhysicalEnergy(nn.Module):
    """E[z] = Q(u) + mean_x phi(u) + mean_x e(s):
       Q      fixed quadratic mechanical energy (paper-1 diagnostic energy; for wave 1/2(v^2 + c^2|grad q|^2)),
       phi>=0 learned pointwise potential on u  (needed for nonlinear reversible transport: Burgers = d_x delta/du int u^3/6),
       e >=0  learned convex, increasing internal energy of the entropy density, so T = e'(s) > 0.
       E is conserved exactly and phi, e >= 0  =>  Q(u_t) <= E[z_0] for all t (bounded physical energy by construction)."""
    def __init__(self, cfg, p):
        super().__init__(); self.cfg = cfg; self.p = p
        self.phi = nn.Sequential(nn.Linear(p, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU(), nn.Linear(32, 1))
        self.w_raw = nn.Parameter(-3.0 * torch.ones(16)); self.a_raw = nn.Parameter(torch.randn(16) * 0.5); self.b = nn.Parameter(torch.randn(16))
    def _quad_basis(self, x):                                            # monomials of degree <= 2 in the p channels
        cols = [torch.ones_like(x[..., 0])] + [x[..., i] for i in range(self.p)]
        cols += [x[..., i] * x[..., j] for i in range(self.p) for j in range(i, self.p)]
        return torch.stack(cols, -1)
    def potential(self, u):                                              # (B, *grid)
        """phi(u) = psi(u) - Pi_quad psi(u):  psi = softplus(MLP) >= 0, minus its weighted least-squares projection onto
        polynomials of degree <= 2, computed on a fixed Gaussian quadrature grid over the data range.  This removes the
        quadratic gauge GLOBALLY (a Taylor subtraction at u=0 does not: the network can hide a quadratic away from the
        origin, and did).  Without it phi absorbs alpha*Q, L's symbol settles at (1-alpha) x exact and the explicit
        remainder step carries the rest.  phi is not sign-definite; the bound reads Q(u_t) <= E(z_0) - |Omega| inf phi."""
        x = u.movedim(1, -1)
        psi = F.softplus(self.phi(x)).squeeze(-1)
        if getattr(self.cfg, 'phi_gauge', 'proj') == 'none':
            return psi
        if not hasattr(self, '_nodes') or self._nodes.device != u.device or self._nodes.dtype != u.dtype:
            g = torch.linspace(-4, 4, 17, device=u.device, dtype=u.dtype)
            nodes = torch.stack(torch.meshgrid(*([g] * self.p), indexing='ij'), -1).reshape(-1, self.p)
            wts = torch.exp(-0.5 * (nodes ** 2).sum(-1))                           # unit-variance Gaussian reference measure
            self._nodes, self._wts = nodes, wts
        Phi = self._quad_basis(self._nodes)                                         # (n, nb)
        psi_n = F.softplus(self.phi(self._nodes)).squeeze(-1)                       # (n,)
        W = self._wts
        A = Phi.T @ (Phi * W.unsqueeze(-1)); bvec = Phi.T @ (psi_n * W)                # weighted normal equations
        coef = torch.linalg.solve(A + 1e-8 * torch.eye(A.shape[0], device=A.device, dtype=A.dtype), bvec)
        return psi - (self._quad_basis(x) * coef).sum(-1)
    T0 = 0.1   # temperature floor: e(s) = T0 s + e_tilde(s), e_tilde convex >= 0, so T = e'(s) >= T0.  Physical friction
               # coefficients carry 1/T; without a floor, regions where s drifts negative send T -> 0 and the explicit
               # dissipative step blows up.  The energy bound survives via the second law (S non-decreasing):
               #   Q(u_t) <= E(z_0) - T0 S(z_0) - |Omega| inf phi.
    def internal(self, s):                                               # (B, *grid), convex increasing in s
        w, a = F.softplus(self.w_raw), F.softplus(self.a_raw)
        return self.T0 * s.squeeze(1) + (w * F.softplus(a * s.unsqueeze(-1) + self.b)).sum(-1).squeeze(1)
    def temperature(self, s):                                            # T = e'(s) >= T0
        w, a = F.softplus(self.w_raw), F.softplus(self.a_raw)
        return self.T0 + (w * a * torch.sigmoid(a * s.unsqueeze(-1) + self.b)).sum(-1)
    def forward(self, z, p):
        Q, _ = phys_energy(z[:, :p], self.cfg)
        return Q + self.potential(z[:, :p]).flatten(1).mean(1) + self.internal(z[:, p:]).flatten(1).mean(1)

class IntegralEntropy(nn.Module):
    """S[z] = mean_x s : the textbook Casimir. dS/dz = (0, 1) exactly."""
    def forward(self, z, p):
        return z[:, p:].flatten(1).mean(1)

class StateDissipativeOperator(DissipativeOperator):
    """M = (I-P_E) A A^T (I-P_E),  A = D_M o P(x) o Gamma(z),  with an ONSAGER PARITY structure.
    Slots (r = c*(d+1)): for each channel i, d 'flux' slots (i,a) and one 'local' slot i.
      D_M   : channel i <- flux slot (j,a) via  -i k_a * b_{ij a}(k)  (a divergence times an even real symbol),
              channel i <- local slot j  via  b'_{ij}(k) (even real symbol).  Derivative parity is fixed by construction.
      Gamma : pointwise r x r.  flux-flux and local-local blocks are arbitrary pointwise functions of z;
              flux<->local blocks are LINEAR in the gradient features (grad z)(x) with pointwise coefficients.
    Hence every path channel <- channel through M has EVEN total derivative order: M can diffuse and damp but cannot
    transport.  Without this the dissipative channel can carry advection (M(0,1)_u = -d_x(c u)), and the learned
    reversible symbol splits the transport with it (observed: exactly 1/2 of the exact a(k), 1/3 of d(k) for wave).
    The closed-form heat / damped-wave friction operators are exactly of this form (verify_representability)."""
    def __init__(self, cfg, c, m, p, init_scale=0.05, conserve_mass=False):
        nn.Module.__init__(self)
        self.dim, self.c, self.m, self.p, self.conserve_mass = cfg.dim, c, m, p, conserve_mass
        d = cfg.dim; self.d = d; self.nf = c * d; self.r = c * d + c
        # even real symbols on the band: b (c, c, d) for flux slots, b' (c, c) for local slots
        self.b = nn.Parameter(init_scale * torch.randn(c, c, d, *([2 * m + 1] * d)))
        self.bl = nn.Parameter(init_scale * torch.randn(c, c, *([2 * m + 1] * d)))
        with torch.no_grad():
            if cfg.flux_init:
                for i in range(c):
                    for a in range(d): self.b[i, i, a] += cfg.flux_scale
                    self.bl[i, i] += cfg.flux_scale
        # channel masks: no local slot into conserved-density channels (mass), no friction at all on a wave's q
        rowmask = torch.ones(c); locmask = torch.ones(c)
        if cfg.pde == 'wave': rowmask[0] = 0.0
        if conserve_mass: locmask[:p] = 0.0
        self.register_buffer('row_mask', rowmask); self.register_buffer('loc_mask', locmask)
        # pointwise coefficient network on z: flux-flux block (nf x nf), local-local block (c x c),
        # cross coefficients h (nf x c x nf) and h' (c x nf x nf) multiplying gradient features
        nff, ncc, nx1, nx2 = self.nf ** 2, c ** 2, self.nf * c * self.nf, c * self.nf * self.nf
        self.coef = nn.Sequential(nn.Linear(c, 64), nn.GELU(), nn.Linear(64, 64), nn.GELU(), nn.Linear(64, nff + ncc + nx1 + nx2))
        nn.init.zeros_(self.coef[-1].weight); nn.init.zeros_(self.coef[-1].bias)
        self.gamma = None; self._cache = {}
    # ---- symbols ----
    def _even(self, prm, nx):
        full = torch.stack([_scatter_band(prm[idx], nx, self.dim) for idx in itertools.product(*[range(n) for n in prm.shape[:-self.dim]])])
        full = full.view(*prm.shape[:-self.dim], *([nx] * self.dim))
        return 0.5 * (full + neg_index(full, self.dim))
    def symbol_B(self, nx):
        """c x r complex symbol of D_M in fftn layout (columns: flux slots (j,a) then local slots j)."""
        key = (nx, str(self.b.device))
        if key not in self._cache:
            k = torch.fft.fftfreq(nx, d=1.0 / nx, device=self.b.device)
            self._cache[key] = torch.meshgrid(*([k] * self.dim), indexing='ij')
        ks = self._cache[key]
        be = self._even(self.b, nx); ble = self._even(self.bl, nx)                    # (c,c,d,*g), (c,c,*g)
        # saturated symbols: |flux symbol| < 1 and |local symbol| < 1 for ANY learned b, b'. Since M = A A^T for any A this
        # keeps symmetry/PSD/degeneracy exactly, and it bounds the gain of the dissipative operator so the explicit
        # sub-stepped remainder cannot go unstable at high k when b(k) drifts (14/72 seeds diverged without it).
        # For the physical heat symbol sqrt(nu) k the saturation is negligible on the data band ((sqrt(nu) k)^2 ~ 0.3 at k=8).
        kb = torch.stack([ks[a] * be[:, :, a] for a in range(self.d)], 2)
        flux = -1j * kb / torch.sqrt(1.0 + kb ** 2)                                     # (c, c, d, *g)
        flux = flux.reshape(self.c, self.nf, *([nx] * self.dim))
        loc = ((ble / torch.sqrt(1.0 + ble ** 2)) * self.loc_mask.view(1, self.c, *([1] * self.dim))).to(flux.dtype)
        B = torch.cat([flux, loc], 1) * self.row_mask.view(self.c, 1, *([1] * self.dim))
        return B
    # ---- state ----
    def grad_features(self, z):                                                          # (B, c*d, *grid): d_a z_j
        d = sdims(self.dim); nx = z.shape[-1]; ks = self._cache[(nx, str(z.device))] if (nx, str(z.device)) in self._cache else None
        if ks is None:
            self.symbol_B(nx); ks = self._cache[(nx, str(z.device))]
        zf = torch.fft.fftn(z, dim=d)
        g = torch.stack([torch.fft.ifftn(1j * ks[a] * zf, dim=d).real for a in range(self.d)], 2)   # (B, c, d, *g)
        return g.reshape(z.shape[0], self.nf, *z.shape[2:])
    def set_state(self, z, gE, gamma=None):
        Bsym = self.symbol_B(z.shape[-1])
        w = self._mul(gE, Bsym.conj().transpose(0, 1))
        self.w_hat = w / (ip_pointwise(w, w).sqrt().unsqueeze(1) + 1e-30)
        if gamma is not None: self.gamma = gamma; return
        B, c, nf, r = z.shape[0], self.c, self.nf, self.r
        co = (torch.tanh(self.coef(z.movedim(1, -1))) / self.r).movedim(-1, 1)           # bounded so |Gamma - I| <= 1:
        # unbounded pointwise coefficients let the s-channel's local dynamics stiffen (s grew 40%/step and blew up the
        # explicit remainder step on advection); physical friction coefficients are bounded functions of the state
        i0 = 0
        ff = co[:, i0:i0 + nf * nf].view(B, nf, nf, *z.shape[2:]); i0 += nf * nf
        ll = co[:, i0:i0 + c * c].view(B, c, c, *z.shape[2:]); i0 += c * c
        h = co[:, i0:i0 + nf * c * nf].view(B, nf, c, nf, *z.shape[2:]); i0 += nf * c * nf
        hp = co[:, i0:i0 + c * nf * nf].view(B, c, nf, nf, *z.shape[2:])
        gz = self.grad_features(z)                                                       # (B, nf, *grid)
        G = torch.zeros(B, r, r, *z.shape[2:], device=z.device, dtype=z.dtype)
        eye_f = torch.eye(nf, device=z.device, dtype=z.dtype).view(1, nf, nf, *([1] * self.dim))
        eye_l = torch.eye(c, device=z.device, dtype=z.dtype).view(1, c, c, *([1] * self.dim))
        G[:, :nf, :nf] = eye_f + ff
        G[:, nf:, nf:] = eye_l + ll
        G[:, :nf, nf:] = torch.einsum('bijk...,bk...->bij...', h, gz)                    # flux <- local: linear in grad z
        G[:, nf:, :nf] = torch.einsum('bijk...,bk...->bij...', hp, gz)                   # local <- flux: linear in grad z
        self.gamma = G
    def _mul(self, v, Bsym):
        d = sdims(self.dim); vf = torch.fft.fftn(v, dim=d)
        return torch.fft.ifftn(torch.einsum('ij...,bj...->bi...', Bsym, vf), dim=d).real
    def _P(self, w):
        return w - ip_pointwise(self.w_hat, w).unsqueeze(1) * self.w_hat
    def raw(self, v):                           # A A^T v = D_M P Gamma Gamma^T P D_M^H v
        Bsym = self.symbol_B(v.shape[-1])
        w = self._P(self._mul(v, Bsym.conj().transpose(0, 1)))
        w = torch.einsum('bji...,bj...->bi...', self.gamma, w)
        w = torch.einsum('bij...,bj...->bi...', self.gamma, w)
        return self._mul(self._P(w), Bsym)
    def forward(self, v, w, project=False):
        if not project: return self.raw(v)
        return self.proj_off(w, self.raw(self.proj_off(w, v)))

class PoissonGENERIC(nn.Module):
    """dz/dt = L dE/dz + M dS/dz  on z = (u, s):  L constant Poisson, S a Casimir of L by construction,
    M PSD with the projection sandwich.  Full metriplectic structure, exact for any parameters."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.p = n_fields(cfg.pde); c = self.p + 1
        self.L = PoissonOperator(cfg.dim, self.p, cfg.m_L, cfg.K_mode, canonical=(cfg.pde == 'wave'), init_scale=cfg.op_init)
        self.L.use_lp = bool(getattr(cfg, 'lie_poisson', False)) and cfg.pde != 'wave' and cfg.energy == 'physical'
        cm = (cfg.pde != 'wave') if cfg.conserve_mass == 'auto' else (cfg.conserve_mass == 'yes')   # scalar conservation laws conserve int u
        self.M = (StateDissipativeOperator(cfg, c, cfg.m_M, self.p, init_scale=cfg.op_init, conserve_mass=cm) if cfg.m_type == 'state'
                  else DissipativeOperator(cfg.dim, c, cfg.m_M, init_scale=cfg.op_init))
        self.E = PhysicalEnergy(cfg, self.p) if cfg.energy == 'physical' else Functional(cfg, c)
        self.S = IntegralEntropy() if cfg.entropy == 'integral' else Functional(cfg, c, proj=self.L.project_K)
        self.enc = make_backbone(cfg, self.p, 1, width=max(cfg.width // 2, 8)) if cfg.s_init == 'encoder' else None
    # ---- state ----
    def init_state(self, u):
        s = self.enc(u) if self.enc is not None else torch.zeros_like(u[:, :1])
        return torch.cat([u, s], 1)
    def Lz(self, g, z=None):                     # L acts on the u-block only; s-block of L is zero
        u = z[:, :self.p] if z is not None else None
        return torch.cat([self.L(g[:, :self.p], u), torch.zeros_like(g[:, self.p:])], 1)
    def Lz_const(self, g):                       # constant-multiplier part of L only (no Lie-Poisson term)
        return torch.cat([self.L(g[:, :self.p], None), torch.zeros_like(g[:, self.p:])], 1)
    def PiK_z(self, d):                          # projection onto ker L  (K modes of u, all of s)
        return torch.cat([self.L.project_K(d[:, :self.p]), d[:, self.p:]], 1)
    def grads(self, z, create_graph=True):
        with torch.enable_grad():
            if not z.requires_grad: z = z.requires_grad_(True)
            E = self.E(z, self.p); S = self.S(z, self.p)
            N = z[0, 0].numel()   # Riesz representative w.r.t. the grid-measure inner product: (#grid points) x the autodiff gradient
            gE = N * torch.autograd.grad(E.sum(), z, create_graph=create_graph)[0]
            gS = N * torch.autograd.grad(S.sum(), z, create_graph=create_graph)[0]
        return E, S, gE, gS
    def vector_field(self, z, create_graph=True):
        E, S, gE, gS = self.grads(z, create_graph)
        if isinstance(self.M, StateDissipativeOperator): self.M.set_state(z, gE)
        f = self.Lz(gE, z) + self.M(gS, gE)
        return f, dict(E=E, S=S, gE=gE, gS=gS)
    # ---- integrators ----
    def step_euler(self, z, create_graph=True):
        f, aux = self.vector_field(z, create_graph)
        return z + f, aux
    def step_dg(self, z, iters=None, create_graph=False):
        """Discrete-gradient (Gonzalez) step. dE/dz uses the standard correction; dS/dz uses the correction
        restricted to ker L, so L gS_bar = 0 still holds and both laws are exact in discrete time (Thm 2)."""
        iters = iters or self.cfg.dg_iters
        # v11b: predictor = the split step (exact linear flow + sub-cycled remainder) when the energy is physical, as A.5 says;
        # the explicit Euler predictor over the full dt diverged on the 2D heat cells. E0, S0 are evaluated at z itself.
        z1, aux = self.step_split(z, create_graph) if isinstance(self.E, PhysicalEnergy) else self.step_euler(z, create_graph)
        E0 = self.E(z, self.p); S0 = self.S(z, self.p)
        for it in range(iters):
            zm = 0.5 * (z + z1); d = z1 - z
            E1, S1 = self.E(z1, self.p), self.S(z1, self.p)
            _, _, gE, gS = self.grads(zm, create_graph)
            if isinstance(self.M, StateDissipativeOperator): self.M.set_state(zm, gEb if it > 0 else gE)
            cE = (E1 - E0 - ip(gE, d)) / (ip(d, d) + 1e-30)
            gEb = gE + cE.view(-1, *([1] * (z.dim() - 1))) * d
            dK = self.PiK_z(d)
            cS = (S1 - S0 - ip(gS, d)) / (ip(dK, dK) + 1e-30)
            gSb = gS + cS.view(-1, *([1] * (z.dim() - 1))) * dK
            z_new = z + self.Lz(gEb, zm) + self.M(gSb, gEb)
            z1 = z_new if it == iters - 1 else 0.5 * (z1 + z_new)     # relaxed iterations, plain final update
        aux.update(dict(gEbar=gEb, gSbar=gSb))
        return z1, aux
    def step_split(self, z, create_graph=True):
        """Strang split: exact unitary half-step of the linear reversible flow (conserves Q exactly), Euler step of the
        remainder (nonlinear potential + dissipation), exact half-step again.  Physical-energy variant only."""
        assert isinstance(self.E, PhysicalEnergy), 'split integrator needs the fixed quadratic energy'
        p = self.p
        u = self.L.exp_linear(z[:, :p], self.cfg, 0.5); z = torch.cat([u, z[:, p:]], 1)
        f, aux = self.vector_field(z, create_graph)
        _, gQ = phys_energy(z[:, :p], self.cfg)                            # linear part of dE/du, already integrated exactly
        f = f - self.Lz_const(torch.cat([gQ, torch.zeros_like(z[:, p:])], 1))   # subtract ONLY the constant-multiplier part
        z = z + f / self.cfg.n_sub
        for _ in range(self.cfg.n_sub - 1):                                # optional sub-cycling of the stiff dissipative remainder
            f, _ = self.vector_field(z, create_graph); _, gQ = phys_energy(z[:, :p], self.cfg)
            z = z + (f - self.Lz_const(torch.cat([gQ, torch.zeros_like(z[:, p:])], 1))) / self.cfg.n_sub
        u = self.L.exp_linear(z[:, :p], self.cfg, 0.5); z = torch.cat([u, z[:, p:]], 1)
        return z, aux
    def step(self, z, create_graph=True, integrator=None):
        integrator = integrator or self.cfg.integrator
        if integrator == 'dg':
            return self.step_dg(z, create_graph=create_graph)
        if integrator == 'split':
            return self.step_split(z, create_graph)
        return self.step_euler(z, create_graph)
    def rollout(self, u0, n, create_graph=True, keep_aux=False, integrator=None):
        z = self.init_state(u0); us, zs, auxs = [], [z], []
        for _ in range(n):
            z, aux = self.step(z, create_graph, integrator)
            if not create_graph: z = z.detach()
            us.append(z[:, :self.p]); zs.append(z)
            if keep_aux: auxs.append({k: v.detach() for k, v in aux.items()})
        return torch.stack(us, 1), torch.stack(zs, 1), auxs

class Baseline(nn.Module):
    """Unconstrained residual operator  u_{t+1} = u_t + G(u_t)."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.p = n_fields(cfg.pde)
        self.net = make_backbone(cfg, self.p, self.p)
    def rollout(self, u0, n, create_graph=True, keep_aux=False, integrator=None):
        u = u0; us = []
        for _ in range(n):
            u = u + self.net(u); us.append(u)
        return torch.stack(us, 1), None, []

def make_model(cfg):
    return (PoissonGENERIC(cfg) if cfg.model == 'pg' else Baseline(cfg)).to(DEV)

In [ ]:
def phys_energy(u, cfg):
    """Fixed quadratic physical energy Q(u) and its gradient dQ/du (no learned quantity involved)."""
    if cfg.pde != 'wave':
        return 0.5 * ip(u, u), u
    sp = Spectral(u.shape[-1], cfg.dim, u.device, u.dtype); c2 = PDE_PARAMS['wave']['c'] ** 2
    q, v = u[:, :1], u[:, 1:]
    Q = 0.5 * (ip(v, v) - c2 * ip(q, sp.lap(q)))
    return Q, torch.cat([-c2 * sp.lap(q), v], 1)

def rel_l2(pred, true):                          # per-sample relative L2, averaged -> scalar
    return (nrm(pred - true) / (nrm(true) + 1e-12)).mean()

def train_model(model, data, cfg, verbose=True):
    op_names = {'L.a', 'L.lam', 'L.alpha', 'M.b', 'M.bl'}   # operator symbols get the large lr; the friction symbols are saturated (|.|<1), so this is safe
    op_params = [p for n, p in model.named_parameters() if n in op_names]
    other = [p for n, p in model.named_parameters() if n not in op_names]
    groups = [dict(params=other, lr=cfg.lr)] + ([dict(params=op_params, lr=cfg.lr_op, weight_decay=0.0)] if op_params else [])
    opt = torch.optim.AdamW(groups, lr=cfg.lr, weight_decay=cfg.wd)
    n_iter = cfg.epochs * math.ceil(data.shape[0] / cfg.batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in groups], total_steps=n_iter, pct_start=0.1)
    H = cfg.horizon; model.train(); t0 = time.time(); hist = []
    for ep in range(cfg.epochs):
        perm = torch.randperm(data.shape[0]); tot = 0.0; nb = 0
        for i in range(0, data.shape[0], cfg.batch):
            traj = data[perm[i:i + cfg.batch]].to(DEV)
            t_start = np.random.randint(0, cfg.T - H + 1)
            u0 = traj[:, t_start]; target = traj[:, t_start + 1:t_start + 1 + H]
            pred, _, _ = model.rollout(u0, H, create_graph=True, integrator=cfg.train_integrator)
            loss = sum(rel_l2(pred[:, h], target[:, h]) for h in range(H)) / H
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot += loss.item(); nb += 1
        hist.append(tot / nb)
        if verbose and (ep % max(1, cfg.epochs // 5) == 0 or ep == cfg.epochs - 1):
            print(f'  ep {ep:3d}  loss {tot/nb:.4f}  ({time.time()-t0:.0f}s)')
    return hist

@torch.no_grad()
def evaluate(model, data, cfg):
    """Rollout from t=0 for cfg.n_roll steps on held-out trajectories. Returns accuracy + structural diagnostics."""
    model.eval(); out = {}
    traj = data.to(DEV); n = min(cfg.n_roll, cfg.T)
    u0 = traj[:, 0]; true = traj[:, 1:1 + n]
    pred, zs, auxs = model.rollout(u0, n, create_graph=False, keep_aux=True)
    errs = [rel_l2(pred[:, t], true[:, t]).item() for t in range(n)]
    out['l2_1step'] = errs[0]; out['l2_rollout'] = float(np.mean(errs)); out['l2_final'] = errs[-1]
    # gauge-invariant dissipation of the FIXED physical energy Q (references no learned quantity):
    #   scalar PDEs: Q = 1/2||u||^2 ;  wave: Q = 1/2(||v||^2 + c^2||grad q||^2)   (Q of the observed field alone
    #   is not an invariant of the wave, cf. GENERIC-FNO Sec. 5.4).  r_mech = -<dQ/du, du> / (||dQ/du|| ||du||).
    prev = torch.cat([u0[:, None], pred[:, :-1]], 1).flatten(0, 1); du = (pred - prev.view_as(pred)).flatten(0, 1)
    # midpoint form: for a quadratic Q, -<dQ/du(u_mid), du> = Q(u) - Q(u') exactly, so an exactly Q-conserving step
    # (e.g. transport) gives r_mech = 0; the endpoint form has an O(|du|/|u|) bias (~0.1 at this frame spacing)
    _, gQ = phys_energy(0.5 * (prev + prev + du), cfg)
    out['rmech'] = (-ip(gQ, du) / (nrm(gQ) * nrm(du) + 1e-12)).mean().item()
    tprev = torch.cat([u0[:, None], true[:, :-1]], 1).flatten(0, 1)
    Qt0, _ = phys_energy(tprev, cfg); Qt1, _ = phys_energy(true.flatten(0, 1), cfg)
    out['Pi_star'] = ((Qt0 - Qt1) / (Qt0 + 1e-12)).mean().item()
    # same normalization as Pi_star (fractional one-step change of the fixed Q), so model and truth compare by MAGNITUDE,
    # not only by rank: r_mech is a cosine and under-weights dissipation on transport-dominated flows (Burgers)
    Qm0, _ = phys_energy(prev, cfg); Qm1, _ = phys_energy(pred.flatten(0, 1), cfg)
    out['Pi_model'] = ((Qm0 - Qm1) / (Qm0 + 1e-12)).mean().item()
    out['Q_ratio_final'] = (phys_energy(pred[:, -1], cfg)[0] / phys_energy(true[:, -1], cfg)[0]).mean().item()
    if isinstance(model, PoissonGENERIC):
        if isinstance(model.E, PhysicalEnergy):    # Theorem: Q(u_t) <= E[z_0] - |Omega| inf phi   (= E[z_0] when phi >= 0)
            E0 = model.E(zs[:, 0], model.p)
            Qmax = torch.stack([phys_energy(pred[:, t], cfg)[0] for t in range(n)], 1).amax(1)
            phi_min = torch.stack([model.E.potential(pred[:, t]).flatten(1).amin(1) for t in range(n)], 1).amin(1).clamp(max=0)
            S0 = zs[:, 0, model.p:].flatten(1).mean(1)
            out['phi_min'] = phi_min.min().item()
            out['Qmax_over_E0'] = (Qmax / E0).max().item()
            out['Qmax_over_bound'] = (Qmax / (E0 - model.E.T0 * S0 - phi_min)).max().item()      # theorem: <= 1
        rE, dE, dS, sgn = [], [], [], []
        for t, aux in enumerate(auxs):
            dz = zs[:, t + 1] - zs[:, t]; g = aux.get('gEbar', aux['gE'])   # DG: test against the discrete gradient
            rE.append((ip(g, dz).abs() / (nrm(g) * nrm(dz) + 1e-30)).mean().item())
            E1, S1, _, _ = model.grads(zs[:, t + 1], create_graph=False)
            dE.append(((E1 - aux['E']).abs() / (aux['E'].abs() + 1e-12)).mean().item())
            dS.append((S1 - aux['S']).min().item())
        out['rE'] = float(np.mean(rE)); out['dE_rel'] = float(np.mean(dE)); out['min_dS'] = float(np.min(dS))
    return out

In [ ]:
def lie_poisson_jacobi_vs_resolution(band=8, sizes=(32, 48, 64, 128), dtype=torch.float64):
    """Jacobi residual of the discrete Lie-Poisson operator u D + D u on band-limited states with band-limited test
    functionals, as the grid is refined at fixed band.  Triple products need 3*band <= nx/2 to be alias-free."""
    torch.set_default_dtype(dtype); rows = []
    for nx in sizes:
        k = torch.fft.fftfreq(nx, d=1.0 / nx, device=DEV, dtype=dtype); mask = (k.abs() <= band)
        D = lambda x: torch.fft.ifft(1j * k * torch.fft.fft(x)).real
        P = lambda x: torch.fft.ifft(mask * torch.fft.fft(x)).real
        L = lambda zz, g: (zz[:, 0] * D(g[:, 0]) + D(zz[:, 0] * g[:, 0])).unsqueeze(1)
        z = P(torch.randn(2, nx, device=DEV, dtype=dtype)).unsqueeze(1)
        fs = []
        for _ in range(3):
            r = torch.stack([P(torch.randn(nx, device=DEV, dtype=dtype)) for _ in range(3)]); al = torch.randn(3, device=DEV, dtype=dtype)
            def fn(zz, r=r, al=al):
                N = zz[0].numel()
                return sum(al[j] * torch.tanh(ip(zz, r[j].view(1, 1, -1).expand_as(zz)) / math.sqrt(N)) for j in range(3)) + 0.5 * ip(zz, zz) / N
            fs.append(fn)
        rows.append(dict(nx=nx, band=band, resolved=(3 * band <= nx // 2), jacobi=jacobi_residual(z, fs, L)))
    torch.set_default_dtype(torch.float32)
    return pd.DataFrame(rows).set_index('nx')

_DATA_CACHE = {}
def get_data(cfg):
    key = (cfg.dim, cfg.pde, cfg.nx, cfg.n_traj, cfg.T, cfg.dt, cfg.kmax, cfg.seed)
    if key not in _DATA_CACHE:
        _DATA_CACHE[key] = gen_data(cfg).cpu()
    return _DATA_CACHE[key]

def run_one(cfg, verbose=True):
    seed_all(cfg.seed + 1)
    data = get_data(cfg); tr, te = split_data(data)
    model = make_model(cfg)
    if verbose: print(f'[{cfg.dim}D {cfg.pde:9s} {cfg.backbone:10s} {cfg.model:4s}] params={count_params(model):,}')
    t0 = time.time(); hist = train_model(model, tr, cfg, verbose)
    row = dict(dim=cfg.dim, pde=cfg.pde, backbone=cfg.backbone, model=cfg.model, seed=cfg.seed,
               params=count_params(model), train_s=time.time() - t0, final_loss=hist[-1])
    row.update(evaluate(model, te, cfg))
    # persistence reference
    te_d = te.to(DEV); n = min(cfg.n_roll, cfg.T)
    row['l2_persist'] = float(np.mean([rel_l2(te_d[:, 0], te_d[:, 1 + t]).item() for t in range(n)]))
    if verbose: print('   ', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items() if k not in ('dim','pde','backbone','model','seed')})
    return row, model

def band_limit(x, band):
    """Project (…, *grid) fields onto |k|_inf <= band."""
    dim = 1 if x.dim() <= 2 else x.dim() - 2 if x.dim() > 3 else 1
    d = tuple(range(-dim, 0)); nx = x.shape[-1]
    k = torch.fft.fftfreq(nx, d=1.0 / nx, device=x.device)
    ks = torch.meshgrid(*([k] * dim), indexing='ij'); mask = torch.stack([kk.abs() for kk in ks]).amax(0) <= band
    return torch.fft.ifftn(torch.fft.fftn(x, dim=d) * mask, dim=d).real

In [ ]:
QUICK = False
SEEDS     = [0] if QUICK else [0, 1, 2]
DIMS      = [1] if QUICK else [1, 2]
PDES      = ['heat', 'advection', 'burgers', 'wave']
BACKBONES = ['fno', 'transolver'] if QUICK else ['fno', 'transolver', 'cno']
VARIANTS = {                                   # name -> cfg overrides
    'base':    dict(model='base'),
    'pg':      dict(model='pg'),                                                        # learned E, Casimir S_phi, constant M
    'pgphys':  dict(model='pg', energy='physical', entropy='integral', m_type='state',   # fixed Q + phi(u) + e(s), S = int s, M(z),
                    integrator='split', train_integrator='split', n_sub=8, lie_poisson=True),   # L = alpha1 D + lambda(uD+Du) (scalar) / J (wave)
    'pgphysC': dict(model='pg', energy='physical', entropy='integral', m_type='state',   # same with a CONSTANT L (exact Jacobi for
                    integrator='split', train_integrator='split', n_sub=8, lie_poisson=False),  # any state; cannot represent Burgers)
}
MODELS = list(VARIANTS)

def cfg_for(dim, pde, backbone, variant, seed):
    if dim == 1:
        c = Cfg(dim=1, pde=pde, nx=64, n_traj=120, T=20, kmax=8, m_L=16, m_M=16, width=24, modes=12, layers=3,
                epochs=15 if QUICK else 60, batch=16, horizon=4, norm=False)
    else:
        c = Cfg(dim=2, pde=pde, nx=32, n_traj=80, T=15, kmax=6, m_L=10, m_M=10, width=24, modes=8, layers=3,
                epochs=10 if QUICK else 40, batch=8, horizon=3, norm=True)
    for k, v in VARIANTS[variant].items(): setattr(c, k, v)
    if variant == 'pg': c.width = 12                    # two functionals + encoder ~ one baseline budget
    if variant.startswith('pgphys'): c.m_M = c.nx // 3           # de-aliased band of the data: Burgers steepens beyond kmax (2D under-dissipated at m_M=kmax)
    c.backbone, c.seed = backbone, seed
    if not variant.startswith('pgphys'): c.integrator = c.train_integrator = 'euler'   # BUG FIX: this line previously
    #   matched 'pgphysC' too, so the constant-L ablation was trained/evaluated with explicit Euler instead of the split step   # consistent; DG evaluated separately in Sec. 10
    return c


In [ ]:
# ---- is the DG non-convergence backbone-specific? 2D heat / FNO / PG-phys, seed 0 ------------------------------------
# On the trained 2D heat Transolver models the relaxed fixed-point solve of the discrete-gradient step diverged (dg3 finite but
# far off the bound, dg6/dg12 NaN) with both the Euler and the split predictor. This trains the same cell with the FNO backbone
# and evaluates split / dg3 / dg6 / dg12. Converged DG rows (dE_rel ~1e-7..1e-8, max Q/bound < 1) => the limitation is
# "trained 2D heat Transolver models"; NaN again => "trained 2D heat models". ~4.5 min on a T4.
import copy, numpy as np, pandas as pd, os, time
CELL = (2, 'heat', 'fno', 'pgphys'); SEEDS_TO_RUN = [0]
rows = []
for sd in SEEDS_TO_RUN:
    cfg = cfg_for(*CELL, sd); t0 = time.time()
    row, mdl = run_one(cfg, verbose=False); data = get_data(cfg); _, te = split_data(data)
    print(f"seed {sd} trained ({time.time()-t0:.0f}s)", flush=True)
    for integ, iters in [('split', 0), ('dg', 3), ('dg', 6), ('dg', 12)]:
        c = copy.deepcopy(cfg); c.integrator = integ; c.dg_iters = max(iters, 1); mdl.cfg = c
        r = evaluate(mdl, te, c); mdl.cfg = cfg
        rows.append(dict(seed=sd, integrator=f'{integ}{iters or ""}', **{k: r[k] for k in ['l2_rollout', 'l2_final', 'Pi_model', 'Pi_star', 'Q_ratio_final', 'Qmax_over_bound', 'dE_rel', 'rE', 'min_dS']}))
        print(f"   {rows[-1]['integrator']:6s} rollout {r['l2_rollout']:.4f}  Q10/Q* {r['Q_ratio_final']:.3f}  max Q/bound {r['Qmax_over_bound']:.3f}  "
              f"dE_rel {r['dE_rel']:.1e}  min dS {r['min_dS']:+.1e}", flush=True)
t = pd.DataFrame(rows)
out = '/content/drive/MyDrive/poisson_generic/split_vs_dg_2Dheat_fno.csv' if os.path.ismount('/content/drive') else 'split_vs_dg_2Dheat_fno.csv'
os.makedirs(os.path.dirname(out) or '.', exist_ok=True); t.to_csv(out, index=False); print('written', out)
display(t.pivot(index='seed', columns='integrator', values=['l2_rollout', 'Qmax_over_bound', 'Q_ratio_final']).round(4))
print("\nRead: converged DG rows here (finite, dE_rel at the fixed-point floor, max Q/bound < 1) mean the non-convergence is\n"
      "specific to the trained Transolver models; NaN again means it is the 2D heat cell. Neither changes a table.")
if False: print(" if a split-step row shows max Q/bound > 1 and the DG rows of the same seed show < 1 with dE_rel at the\n"
      "fixed-point tolerance, the bound fails only through the split step's energy error, which is what Theorem 2 and Sec. 5 say.\n"
      "If no seed leaves the bound this time, the grid-run failure was seed- and hardware-specific; report the grid number and say so.")


seed 0 trained (269s)
   split  rollout 0.0390  Q10/Q* 1.028  max Q/bound 0.334  dE_rel 9.2e-05  min dS +1.8e-02
   dg3    rollout 0.0391  Q10/Q* 1.029  max Q/bound 0.334  dE_rel 1.0e-06  min dS +1.8e-02
   dg6    rollout 0.0391  Q10/Q* 1.029  max Q/bound 0.334  dE_rel 9.5e-08  min dS +1.8e-02
   dg12   rollout 0.0391  Q10/Q* 1.029  max Q/bound 0.334  dE_rel 1.5e-07  min dS +1.8e-02
written /content/drive/MyDrive/poisson_generic/split_vs_dg_2Dheat_fno.csv


l2_rollout                        Qmax_over_bound                  \
integrator       dg12     dg3     dg6  split            dg12     dg3     dg6   
seed                                                                           
0              0.0391  0.0391  0.0391  0.039          0.3336  0.3336  0.3336   

                   Q_ratio_final                          
integrator   split          dg12     dg3     dg6   split  
seed                                                      
0           0.3337        1.0285  1.0286  1.0285  1.0283


Read: converged DG rows here (finite, dE_rel at the fixed-point floor, max Q/bound < 1) mean the non-convergence is
specific to the trained Transolver models; NaN again means it is the 2D heat cell. Neither changes a table.
